In [ ]:
from google.colab import drive
import os, sys, warnings, math, pathlib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.interpolate import griddata
from scipy.ndimage import generic_filter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_curve, auc, roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.neighbors import KDTree
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({'font.size': 10, 'font.family': 'serif'})
np.random.seed(0)


RAW_DATA_DIR = "./data/raw"

all_files = os.listdir(RAW_DATA_DIR)
dat_files = [os.path.join(RAW_DATA_DIR, f) for f in all_files if f.lower().endswith(('.dat','.xyz','.txt'))]
grd_files = [os.path.join(RAW_DATA_DIR, f) for f in all_files if f.lower().endswith('.grd')]
xls_files = [os.path.join(RAW_DATA_DIR, f) for f in all_files if f.lower().endswith(('.xlsx','.xls'))]

if len(dat_files)==0 and len(grd_files)==0:
    raise SystemExit("No .dat/.xyz/.txt/.grd files found.")

print(f"Found {len(dat_files)} grid files and {len(xls_files)} Excel files")

def read_xyz_dat(fn):
    data = np.loadtxt(fn)
    if data.ndim==1 and data.size==3:
        data = data.reshape(1,3)
    return data[:,0].astype(float), data[:,1].astype(float), data[:,2].astype(float)

def moving_std(arr, size=9):
    return generic_filter(arr, np.nanstd, size=size, mode='nearest')

def dms_to_dec(s):
    if isinstance(s,(int,float)) and not np.isnan(s): return float(s)
    if pd.isna(s): return np.nan
    s = str(s).strip().replace('°',' ').replace("'",' ').replace('"',' ')
    parts = s.split(); nums=[]; dirc=1
    for p in parts:
        if any(c.isalpha() for c in p):
            if 's' in p.lower() or 'w' in p.lower(): dirc=-1
        cleaned=''.join([c for c in p if c.isdigit() or c=='.'])
        if cleaned!='': nums.append(float(cleaned))
    if len(nums)==3: dec = nums[0] + nums[1]/60.0 + nums[2]/3600.0
    elif len(nums)==2: dec = nums[0] + nums[1]/60.0
    elif len(nums)==1: dec = nums[0]
    else: return np.nan
    return dec * dirc

def make_grid(all_x, all_y, grid_res=800):
    xmin,xmax=np.nanmin(all_x),np.nanmax(all_x)
    ymin,ymax=np.nanmin(all_y),np.nanmax(all_y)
    if np.isclose(xmin,xmax) or np.isclose(ymin,ymax):
        raise RuntimeError("Domain extent is zero in X or Y")
    dx,dy = xmax-xmin, ymax-ymin
    if dx >= dy:
        nx = grid_res
        ny = max(4, int(round(grid_res * (dy/dx))))
    else:
        ny = grid_res
        nx = max(4, int(round(grid_res * (dx/dy))))
    xi = np.linspace(xmin, xmax, nx)
    yi = np.linspace(ymin, ymax, ny)
    XI, YI = np.meshgrid(xi, yi)
    return XI, YI, xi, yi

def grid_from_xyz(x,y,z,XI,YI,method='linear'):
    pts = np.column_stack([x,y])
    ZI = griddata(pts, z, (XI,YI), method=method)
    mask_nan = np.isnan(ZI)
    if np.any(mask_nan):
        ZI[mask_nan] = griddata(pts, z, (XI[mask_nan], YI[mask_nan]), method='nearest')
    return ZI

def save_esri_ascii_grid(filename, XI, YI, Z, nodata=-99999):
    ny,nx = Z.shape
    x = XI[0,:]
    y = YI[:,0]
    cellx = x[1]-x[0] if len(x)>1 else 1.0
    celly = y[1]-y[0] if len(y)>1 else 1.0
    xllcorner = x[0] - cellx/2.0
    yllcorner = y[0] - celly/2.0
    with open(filename,'w') as f:
        f.write(f"ncols         {nx}\n")
        f.write(f"nrows         {ny}\n")
        f.write(f"xllcorner     {xllcorner:.6f}\n")
        f.write(f"yllcorner     {yllcorner:.6f}\n")
        f.write(f"cellsize      {cellx:.6f}\n")
        f.write(f"NODATA_value  {nodata}\n")
        for i in range(ny-1, -1, -1):
            row = Z[i,:]
            row_f = [f"{(nodata if np.isnan(v) else v):.6f}" for v in row]
            f.write(" ".join(row_f) + "\n")

os.makedirs("outputs", exist_ok=True)

mapping = {
    'wedge_thick':['accretionary','wedge','wedge_thick','accretionary-wedge'],
    'bathymetry':['bath','bathe','bathymetry'],
    'bulk_density':['bulk','density'],
    'cum_energy':['energy','cumulative','cumulative energy','cumulative_energy'],
    'magnetic':['magnetic','mag'],
    'residual_grav':['residual','gravity','residual_grav'],
    'shear_stress':['shear','stress'],
    'slab_dip':['slab dip','slab_dip','dip'],
    'slab_strike':['strike'],
    'slab_temp':['tempret','temp','temperature','slab temp','slab_temp'],
    'slab_thick':['slab thickness','slab_thick','thickness (km)'],
    'slab_top':['slab top','slab_top','top surface','slab top surface'],
    'taper':['taper','taper angle','taper_angle']
}

found = {}
for fn_path in dat_files + grd_files:
    fn = os.path.basename(fn_path)
    low = fn.lower()
    for key, pats in mapping.items():
        if key in found: continue
        for p in pats:
            if p in low:
                found[key] = fn_path
                break

print("\nAuto-mapped predictors:")
for k,v in found.items():
    print(f" {k:12}-> {os.path.basename(v)}")

grids_xyz = {}
all_x = []; all_y = []
for key,fn_path in found.items():
    x,y,z = read_xyz_dat(fn_path)
    grids_xyz[key] = (x,y,z)
    all_x.append(x); all_y.append(y)

if len(all_x)==0:
    raise SystemExit("No valid grids read.")

all_x = np.concatenate(all_x); all_y = np.concatenate(all_y)
XI, YI, xi, yi = make_grid(all_x, all_y, grid_res=800)
print("Grid shape (ny,nx):", XI.shape)

grid_layers = {}
for key,(x,y,z) in grids_xyz.items():
    try:
        grid_layers[key] = grid_from_xyz(x,y,z,XI,YI)
        print("Loaded", mapping.get(key,[key])[0], "->", key, f"({len(x)} pts)")
    except Exception as e:
        print("Failed to grid", key, e)

if 'magnetic' in grid_layers:
    grid_layers['magnetic_sd'] = moving_std(grid_layers['magnetic'], size=9)

if 'taper' in grid_layers:
    grid_layers['taper_dev'] = np.abs(grid_layers['taper'] - np.nanmedian(grid_layers['taper']))

for key in ['bulk_density','residual_grav','slab_top']:
    if key in grid_layers:
        arr = grid_layers[key]
        grid_layers[f'{key}_inv'] = 1.0 - (arr - np.nanmin(arr)) / (np.nanmax(arr) - np.nanmin(arr) + 1e-12)

if len(xls_files)==0:
    raise SystemExit("No presence Excel found.")

pres_df = pd.read_excel(xls_files[0])
print("Loaded presence Excel:", os.path.basename(xls_files[0]))
print("Columns:", list(pres_df.columns))

lat_col, lon_col = None, None
for c in pres_df.columns:
    lc = c.lower()
    if 'lat' in lc: lat_col = c
    if 'lon' in lc or 'long' in lc: lon_col = c

if lat_col is None or lon_col is None:
    lat_col = pres_df.columns[0]; lon_col = pres_df.columns[1]

pres_x = np.array([dms_to_dec(v) for v in pres_df[lon_col].values])
pres_y = np.array([dms_to_dec(v) for v in pres_df[lat_col].values])
print("First 5 pres coords (lon,lat):", np.column_stack([pres_x,pres_y])[:5])

grid_pts = np.column_stack([XI.ravel(), YI.ravel()])
tree = KDTree(grid_pts)
dists, idx = tree.query(np.column_stack([pres_x, pres_y]), k=1)
ncols = XI.shape[1]
pix_i = (idx // ncols).astype(int).ravel()
pix_j = (idx % ncols).astype(int).ravel()
presence_pixels = np.column_stack([pix_i, pix_j])
print("Presence count mapped to grid:", presence_pixels.shape[0])

n_pres = presence_pixels.shape[0]
n_bg = max(500, 3 * n_pres)
mask_bg = np.isfinite(XI)

for r,c in presence_pixels:
    rr_min = max(0, r-5); rr_max = min(mask_bg.shape[0], r+6)
    cc_min = max(0, c-5); cc_max = min(mask_bg.shape[1], c+6)
    mask_bg[rr_min:rr_max, cc_min:cc_max] = False

bg_idx = np.column_stack(np.nonzero(mask_bg))
if len(bg_idx)==0:
    raise SystemExit("No background cells found.")

sel_bg_idx = bg_idx[np.random.choice(len(bg_idx), size=min(n_bg, len(bg_idx)), replace=False)]
print("Background samples selected:", sel_bg_idx.shape[0])

predictor_template = [
    'slab_temp','slab_top_inv','slab_thick','slab_dip','wedge_thick',
    'residual_grav_inv','bulk_density_inv','magnetic_sd','shear_stress',
    'cum_energy','taper_dev'
]

predictor_keys = [k for k in predictor_template if k in grid_layers]
print("Predictors used:", predictor_keys)

def sample_at_points(keys, rows, cols):
    data=[]
    for r,c in zip(rows,cols):
        row=[]
        for k in keys:
            arr = grid_layers[k]
            val = arr[r,c] if np.isfinite(arr[r,c]) else np.nan
            row.append(val)
        data.append(row)
    return np.array(data, dtype=float)

X_pres = sample_at_points(predictor_keys, presence_pixels[:,0], presence_pixels[:,1])
y_pres = np.ones(X_pres.shape[0], dtype=int)
X_bg = sample_at_points(predictor_keys, sel_bg_idx[:,0], sel_bg_idx[:,1])
y_bg = np.zeros(X_bg.shape[0], dtype=int)

X_all = np.vstack([X_pres, X_bg]); y_all = np.concatenate([y_pres, y_bg])
print("Training samples:", X_all.shape, "Pos:", int(np.sum(y_all==1)), "Bg:", int(np.sum(y_all==0)))

for i in range(X_all.shape[1]):
    col = X_all[:,i]
    col[np.isnan(col)] = np.nanmedian(col)
    X_all[:,i] = col

X_min = np.min(X_all, axis=0); X_max = np.max(X_all, axis=0)
X_scaled = (X_all - X_min) / (X_max - X_min + 1e-12)

coords_all = np.vstack([
    np.column_stack([XI[presence_pixels[:,0], presence_pixels[:,1]], YI[presence_pixels[:,0], presence_pixels[:,1]]]),
    np.column_stack([XI[sel_bg_idx[:,0], sel_bg_idx[:,1]], YI[sel_bg_idx[:,0], sel_bg_idx[:,1]]])
])

kfold=5
km = KMeans(n_clusters=kfold, random_state=0).fit(coords_all)
groups = km.labels_
gkf = GroupKFold(n_splits=kfold)
print("Using spatial KMeans grouping k=", kfold)

prevalence = float(np.sum(y_all==1))/len(y_all)
print(f"Overall positive prevalence in training set: {prevalence:.4f} ({int(np.sum(y_all==1))}/{len(y_all)})")

rf_aucs = []; rf_aps = []
log_aucs = []; log_aps = []
rand_aps = []
fold_counts = []
fold_rocs = []
fold_prs = []

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_scaled, y_all, groups), start=1):
    Xtr, Xte = X_scaled[train_idx], X_scaled[test_idx]
    ytr, yte = y_all[train_idx], y_all[test_idx]
    ntr_pos = int(np.sum(ytr==1)); ntr_neg=int(np.sum(ytr==0))
    nte_pos = int(np.sum(yte==1)); nte_neg=int(np.sum(yte==0))
    fold_counts.append({'fold':fold_idx, 'train_pos':ntr_pos, 'train_neg':ntr_neg, 'test_pos':nte_pos, 'test_neg':nte_neg})
    print(f"\nFold {fold_idx}: train pos/neg = {ntr_pos}/{ntr_neg}, test pos/neg = {nte_pos}/{nte_neg}")

    rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
    rf.fit(Xtr, ytr)
    probs_rf = rf.predict_proba(Xte)[:,1]
    auc_rf = roc_auc_score(yte, probs_rf) if len(np.unique(yte))>1 else np.nan
    ap_rf = average_precision_score(yte, probs_rf)
    rf_aucs.append(auc_rf); rf_aps.append(ap_rf)

    if len(np.unique(yte))>1:
        fpr, tpr, _ = roc_curve(yte, probs_rf)
        fold_rocs.append((fpr,tpr))
    else:
        fold_rocs.append((np.array([0.,1.]), np.array([0.,1.])))

    rec, prec, _ = precision_recall_curve(yte, probs_rf)
    fold_prs.append((rec, prec))
    print(f" RF: ROC-AUC={np.nanmean(auc_rf):.3f}  AP={ap_rf:.3f}")

    log = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=0)
    log.fit(Xtr, ytr)
    probs_log = log.predict_proba(Xte)[:,1]
    auc_log = roc_auc_score(yte, probs_log) if len(np.unique(yte))>1 else np.nan
    ap_log = average_precision_score(yte, probs_log)
    log_aucs.append(auc_log); log_aps.append(ap_log)
    print(f" Logistic: ROC-AUC={np.nanmean(auc_log):.3f}  AP={ap_log:.3f}")

    rnd = np.random.RandomState(42)
    rnd_probs = rnd.uniform(size=len(yte))
    rand_aps.append(average_precision_score(yte, rnd_probs))

    pi = permutation_importance(rf, Xte, yte, n_repeats=30, scoring='average_precision', n_jobs=-1, random_state=0)
    if fold_idx==1:
        perm_stack = np.zeros((len(predictor_keys), kfold))
    perm_stack[:,fold_idx-1] = pi.importances_mean

def meanstd(a): return np.nanmean(a), np.nanstd(a)

print("\n--- Cross-validated summary ---")
print("RF ROC-AUC mean±std:", *meanstd(rf_aucs))
print("RF PR-AUC (AP) mean±std:", *meanstd(rf_aps))
print("Logistic ROC-AUC mean±std:", *meanstd(log_aucs))
print("Logistic PR-AUC mean±std:", *meanstd(log_aps))
print("Random-uniform AP mean±std:", *meanstd(rand_aps))

print("\nFold counts (train/test positives):")
for fc in fold_counts: print(fc)

print("\nMedian RF AP across folds:", np.nanmedian(rf_aps))

def select_keys_by_substrings(keys, substrings):
    return [k for k in keys if any(s in k for s in substrings)]

slab_keys = select_keys_by_substrings(predictor_keys, ['slab_'])
wedge_keys = select_keys_by_substrings(predictor_keys, ['wedge','bulk_density','residual_grav'])
mech_keys = [k for k in predictor_keys if k not in slab_keys + wedge_keys]

print("\nAblation groups:")
print(" slab:", slab_keys)
print(" wedge:", wedge_keys)
print(" mech:", mech_keys)

def crossval_perf_for_keys(selected_keys):
    if len(selected_keys)==0: return np.nan, np.nan
    idxs = [predictor_keys.index(k) for k in selected_keys]
    Xs = X_scaled[:, idxs]
    aucs=[]; aps=[]
    for tr,te in gkf.split(Xs, y_all, groups):
        rf = RandomForestClassifier(n_estimators=200, random_state=0, class_weight='balanced', n_jobs=-1)
        rf.fit(Xs[tr], y_all[tr])
        probs = rf.predict_proba(Xs[te])[:,1]
        aucs.append(roc_auc_score(y_all[te], probs) if len(np.unique(y_all[te]))>1 else np.nan)
        aps.append(average_precision_score(y_all[te], probs))
    return np.nanmean(aucs), np.nanmean(aps)

full_auc, full_ap = meanstd(rf_aucs)[0], meanstd(rf_aps)[0]
slab_auc, slab_ap = crossval_perf_for_keys(slab_keys)
wedge_auc, wedge_ap = crossval_perf_for_keys(wedge_keys)
mech_auc, mech_ap = crossval_perf_for_keys(mech_keys)

print("\nAblation results (ROC, AP):")
print(" Full model:", f"{full_auc:.3f}", f"{full_ap:.3f}")
print(" Slab-only:", f"{slab_auc:.3f}", f"{slab_ap:.3f}")
print(" Wedge-only:", f"{wedge_auc:.3f}", f"{wedge_ap:.3f}")
print(" Mechanical-only:", f"{mech_auc:.3f}", f"{mech_ap:.3f}")

perm_df = []
perm_means = perm_stack.mean(axis=1)
perm_stds = perm_stack.std(axis=1)
for k, m, s in zip(predictor_keys, perm_means, perm_stds):
    perm_df.append({'predictor':k, 'perm_mean': float(m), 'perm_std': float(s)})

perm_df = pd.DataFrame(perm_df).sort_values('perm_mean', ascending=False)
perm_df.to_csv("outputs/feature_permutation_importance.csv", index=False)
print("\nPermutation importance saved to outputs/feature_permutation_importance.csv")
print(perm_df)

n_boot = 50
stack_arrays = []
for i,k in enumerate(predictor_keys):
    arr = grid_layers[k]
    arr_scaled = (arr - X_min[i])/(X_max[i] - X_min[i] + 1e-12)
    arr_scaled[np.isnan(arr_scaled)] = 0.0
    stack_arrays.append(arr_scaled)

fullX = np.vstack([arr.ravel() for arr in stack_arrays]).T
prob_maps = np.zeros((n_boot, XI.shape[0], XI.shape[1]), dtype=np.float32)

print("\nTraining ensemble and predicting full grid ...")
for b in range(n_boot):
    seed = 1000 + b
    idxs = np.random.choice(X_scaled.shape[0], size=X_scaled.shape[0], replace=True)
    clf = RandomForestClassifier(n_estimators=200, random_state=seed, class_weight='balanced', n_jobs=-1)
    clf.fit(X_scaled[idxs], y_all[idxs])
    probs_flat = clf.predict_proba(fullX)[:,1]
    prob_maps[b] = probs_flat.reshape(XI.shape)
    if (b+1) % 10 == 0: print(" Bootstrap", b+1, "done")

mean_prob = np.nanmean(prob_maps, axis=0)
sd_prob = np.nanstd(prob_maps, axis=0)

if 'bathymetry' in grid_layers:
    mask_off = (grid_layers['bathymetry'] < 0) & np.isfinite(grid_layers['bathymetry'])
    mean_prob_mask = np.where(mask_off, mean_prob, np.nan)
    sd_prob_mask = np.where(mask_off, sd_prob, np.nan)
else:
    mean_prob_mask = mean_prob
    sd_prob_mask = sd_prob

XIr = XI.ravel(); YIr = YI.ravel()
np.savetxt("outputs/MDI_mean.dat", np.column_stack([XIr, YIr, mean_prob.ravel()]), fmt="%.6f")
np.savetxt("outputs/MDI_sd.dat", np.column_stack([XIr, YIr, sd_prob.ravel()]), fmt="%.6f")
save_esri_ascii_grid("outputs/MDI_mean.grd", XI, YI, mean_prob_mask, nodata=-99999)
save_esri_ascii_grid("outputs/MDI_sd.grd", XI, YI, sd_prob_mask, nodata=-99999)

print("\nSaved mean and SD maps to outputs/MDI_mean.dat, outputs/MDI_sd.dat, outputs/MDI_mean.grd, outputs/MDI_sd.grd")

summary = {
    'RF_ROC_mean': float(np.nanmean(rf_aucs)), 'RF_ROC_std': float(np.nanstd(rf_aucs)),
    'RF_AP_mean': float(np.nanmean(rf_aps)), 'RF_AP_std': float(np.nanstd(rf_aps)),
    'Log_ROC_mean': float(np.nanmean(log_aucs)), 'Log_AP_mean': float(np.nanmean(log_aps)),
    'Random_AP_mean': float(np.nanmean(rand_aps)), 'Prevalence': float(prevalence),
    'n_pres': int(np.sum(y_all==1)), 'n_bg': int(np.sum(y_all==0)),
    'median_RF_AP': float(np.nanmedian(rf_aps))
}
pd.DataFrame([summary]).to_csv("outputs/model_summary.csv", index=False)
print("Saved outputs/model_summary.csv")

common_fpr = np.linspace(0,1,201)
tprs = []
for fpr,tpr in fold_rocs:
    tprs.append(np.interp(common_fpr, fpr, tpr))
tprs = np.array(tprs)
mean_tpr = np.nanmean(tprs, axis=0)
std_tpr = np.nanstd(tprs, axis=0)
mean_auc = np.nanmean(rf_aucs)

plt.figure(figsize=(6.5,5))
plt.plot(common_fpr, mean_tpr, lw=2, label=f'Mean ROC (AUC={mean_auc:.2f})')
plt.fill_between(common_fpr, np.maximum(mean_tpr-std_tpr,0), np.minimum(mean_tpr+std_tpr,1), alpha=0.25, label='±1 std')
plt.plot([0,1],[0,1], '--', lw=1, color='gray')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('Spatial CV ROC (RF) — mean ± std')
plt.legend(loc='lower right')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("outputs/ROC_mean_std.png", dpi=300)

common_rec = np.linspace(0,1,201)
precisions_interp = []
for rec, prec in fold_prs:
    precisions_interp.append(np.interp(common_rec, rec[::-1], prec[::-1]))
precisions_interp = np.array(precisions_interp)
mean_prec = np.nanmean(precisions_interp, axis=0)
std_prec = np.nanstd(precisions_interp, axis=0)
mean_ap = np.nanmean(rf_aps)

plt.figure(figsize=(6.5,5))
plt.plot(common_rec, mean_prec, lw=2, label=f'Mean PR (AP={mean_ap:.2f})')
plt.fill_between(common_rec, np.maximum(mean_prec-std_prec,0), np.minimum(mean_prec+std_prec,1), alpha=0.25, label='±1 std')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Spatial CV Precision–Recall (RF) — mean ± std')
plt.legend(loc='upper right')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("outputs/PR_mean_std.png", dpi=300)

plt.figure(figsize=(6.5,4))
plt.barh(perm_df['predictor'][::-1], perm_df['perm_mean'][::-1], xerr=perm_df['perm_std'][::-1], align='center')
plt.xlabel('Permutation importance (Δ AP)')
plt.title('Feature permutation importance (mean ± std across folds)')
plt.tight_layout()
plt.savefig("outputs/perm_importance.png", dpi=300)

fig, axes = plt.subplots(1,2,figsize=(10,4))
im0 = axes[0].imshow(mean_prob_mask, origin='lower', aspect='auto')
axes[0].set_title('MDI mean (ensemble)')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)
im1 = axes[1].imshow(sd_prob_mask, origin='lower', aspect='auto')
axes[1].set_title('MDI SD (ensemble)')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)
plt.tight_layout()
plt.savefig("outputs/MDI_mean_sd_preview.png", dpi=300)

print("\nSaved plots: outputs/ROC_mean_std.png, outputs/PR_mean_std.png, outputs/perm_importance.png, outputs/MDI_mean_sd_preview.png")
print("\nAll outputs are in the outputs/ directory.")